# 📘 JSON Handling for Data Engineering
## Databricks Notebook — Semi-Structured Data Mastery

**What you'll learn:**
- Python json module: loads, dumps, nested navigation
- PySpark JSON functions: from_json, get_json_object, explode
- Flattening deeply nested JSON into relational columns
- JSON schema validation and evolution
- VARIANT type (Databricks 2024+)
- JSONL streaming, performance (orjson), building API payloads

**Key rules:**
- Always try/except on json.loads() — malformed JSON is common
- Never use eval() to parse JSON
- Prefer from_json() with explicit schema over schema_of_json()

**Prerequisites:** Notebooks 01-14 (techmart_dw with JSON columns)

---

---
# 📗 Section 1: JSON in Data Engineering

**JSON is everywhere:**
- API responses (REST, GraphQL)
- Event streams (Kafka, Kinesis)
- Configuration files
- NoSQL exports (MongoDB, DynamoDB)
- Semi-structured columns in Delta tables

**Challenge:** JSON has no enforced schema. The same field can be a string in one record
and an object in another. Your pipeline must handle this gracefully.

---
# 📗 Section 2: Python json Module

In [0]:
import json

# json.loads() — string to dict (ALWAYS wrap in try/except)
json_string = '{"customer_id": 1, "name": "John Doe", "orders": [101, 102, 103]}'

try:
    data = json.loads(json_string)
    print(f"Parsed: {data}")
    print(f"Type: {type(data)}")
    print(f"Name: {data['name']}")
    print(f"Orders: {data['orders']}")
except json.JSONDecodeError as e:
    print(f"Invalid JSON: {e}")

# json.dumps() — dict to string
output = json.dumps(data, indent=2, sort_keys=True)
print(f"\nFormatted JSON:\n{output}")

In [0]:
# Handling non-serializable types (datetime, Decimal)
from datetime import datetime
from decimal import Decimal

record = {
    "id": 1,
    "amount": Decimal("150.99"),
    "created_at": datetime(2024, 6, 15, 10, 30),
    "tags": ["premium", "active"]
}

# This would fail: json.dumps(record)
# Fix: use default=str
safe_json = json.dumps(record, default=str)
print(f"With default=str: {safe_json}")

# Better: custom encoder
class DEEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, datetime):
            return obj.isoformat()
        if isinstance(obj, Decimal):
            return float(obj)
        return super().default(obj)

better_json = json.dumps(record, cls=DEEncoder)
print(f"Custom encoder: {better_json}")

---
# 📗 Section 3: Navigating Nested JSON

In [0]:
# Safe nested access with .get()
api_response = {
    "data": {
        "customer": {
            "id": 123,
            "address": {
                "street": "123 Main St",
                "city": "Austin",
                "state": "TX"
            },
            "orders": [
                {"id": 1, "total": 150.0},
                {"id": 2, "total": 250.0}
            ]
        }
    },
    "meta": {"page": 1, "total": 100}
}

# UNSAFE: api_response["data"]["customer"]["address"]["zip"]  → KeyError!
# SAFE: chain .get() with defaults
city = api_response.get("data", {}).get("customer", {}).get("address", {}).get("city", "Unknown")
zip_code = api_response.get("data", {}).get("customer", {}).get("address", {}).get("zip", "N/A")
print(f"City: {city}")
print(f"Zip: {zip_code} (missing field handled gracefully)")

In [0]:
# Recursive flattening
def flatten_json(obj, parent_key="", sep="."):
    """Recursively flatten nested JSON into dot-notation keys."""
    items = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            new_key = f"{parent_key}{sep}{k}" if parent_key else k
            items.extend(flatten_json(v, new_key, sep).items())
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            new_key = f"{parent_key}{sep}{i}" if parent_key else str(i)
            items.extend(flatten_json(v, new_key, sep).items())
    else:
        items.append((parent_key, obj))
    return dict(items)

flat = flatten_json(api_response)
print("Flattened:")
for k, v in flat.items():
    print(f"  {k}: {v}")

In [0]:
# ============================================
# 🎯 YOUR TURN: Extract Nested Values
# ============================================
# Given this JSON (simulating a payment gateway response):
gateway_json = '''
{
  "transaction": {
    "id": "txn_abc123",
    "amount": {"value": 99.99, "currency": "USD"},
    "status": "approved",
    "metadata": {
      "gateway": "stripe",
      "risk_score": 15,
      "flags": ["3ds_verified", "recurring"]
    }
  },
  "customer": {"id": "cust_456", "email": "test@example.com"}
}
'''
# Extract: transaction_id, amount_value, currency, gateway, risk_score, first_flag, customer_email
# Use safe .get() access (no KeyError if field missing)
#
# Write your code below:


In [0]:
# ============================================
# ✅ SOLUTION
# ============================================
import json

data = json.loads(gateway_json)
txn = data.get("transaction", {})

result = {
    "transaction_id": txn.get("id"),
    "amount_value": txn.get("amount", {}).get("value"),
    "currency": txn.get("amount", {}).get("currency"),
    "gateway": txn.get("metadata", {}).get("gateway"),
    "risk_score": txn.get("metadata", {}).get("risk_score"),
    "first_flag": (txn.get("metadata", {}).get("flags") or [None])[0],
    "customer_email": data.get("customer", {}).get("email"),
}

for k, v in result.items():
    print(f"  {k}: {v}")

---
# 📗 Section 5: PySpark JSON Functions

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark.sql("USE techmart_dw")

# get_json_object: extract single field by path
customers = spark.table("customers")
customers.select(
    "customer_id",
    "metadata",
    get_json_object("metadata", "$.source").alias("source"),
    get_json_object("metadata", "$.loyalty_tier").alias("loyalty_tier")
).filter("metadata IS NOT NULL").show(5, truncate=False)

In [0]:
# from_json: parse JSON string into struct (with explicit schema)
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Define schema for the metadata JSON
metadata_schema = StructType([
    StructField("source", StringType(), True),
    StructField("loyalty_tier", IntegerType(), True)
])

parsed = (customers
    .withColumn("meta_struct", from_json("metadata", metadata_schema))
    .withColumn("source", col("meta_struct.source"))
    .withColumn("loyalty_tier", col("meta_struct.loyalty_tier"))
)
parsed.select("customer_id", "source", "loyalty_tier").show(10)

In [0]:
# Parse payments gateway_response JSON
payments = spark.table("payments")

gateway_schema = StructType([
    StructField("gateway", StringType(), True),
    StructField("response_code", StringType(), True),
    StructField("auth_code", StringType(), True)
])

parsed_payments = (payments
    .withColumn("gw", from_json("gateway_response", gateway_schema))
    .withColumn("gateway_name", col("gw.gateway"))
    .withColumn("response_code", col("gw.response_code"))
    .withColumn("auth_code", col("gw.auth_code"))
)
parsed_payments.select("payment_id", "gateway_name", "response_code", "auth_code").show(5)

In [0]:
# ============================================
# 🎯 YOUR TURN: Parse Customer Metadata
# ============================================
# 1. Read the customers table
# 2. Parse the 'metadata' JSON column using from_json with a schema
# 3. Extract 'source' and 'loyalty_tier' into separate columns
# 4. Group by source and calculate avg loyalty_tier
# 5. Show results
#
# Write your code below:


In [0]:
# ============================================
# ✅ SOLUTION
# ============================================
from pyspark.sql.functions import *
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema = StructType([
    StructField("source", StringType()),
    StructField("loyalty_tier", IntegerType())
])

result = (spark.table("customers")
    .withColumn("parsed", from_json("metadata", schema))
    .withColumn("source", col("parsed.source"))
    .withColumn("loyalty_tier", col("parsed.loyalty_tier"))
    .groupBy("source")
    .agg(count("*").alias("customers"), round(avg("loyalty_tier"), 2).alias("avg_tier"))
    .orderBy("customers", ascending=False)
)
result.show()

---
# 📗 Section 6: Flattening Nested JSON in PySpark

In [0]:
# Flatten arrays within JSON using explode
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Create sample nested JSON data
nested_data = [
    (1, '{"name": "John", "orders": [{"id": 101, "amount": 50}, {"id": 102, "amount": 75}]}'),
    (2, '{"name": "Jane", "orders": [{"id": 201, "amount": 120}]}'),
]
df = spark.createDataFrame(nested_data, ["customer_id", "json_data"])

# Define schema
schema = StructType([
    StructField("name", StringType()),
    StructField("orders", ArrayType(StructType([
        StructField("id", IntegerType()),
        StructField("amount", IntegerType())
    ])))
])

# Parse and explode
flattened = (df
    .withColumn("parsed", from_json("json_data", schema))
    .withColumn("name", col("parsed.name"))
    .withColumn("order", explode("parsed.orders"))
    .select("customer_id", "name", col("order.id").alias("order_id"), col("order.amount"))
)
flattened.show()

---
# 📗 Section 7: VARIANT Type (Databricks 2024+)

**VARIANT** is a schema-agnostic type for semi-structured data. Query JSON without parsing.

```sql
-- VARIANT allows dot-notation queries without from_json()
SELECT data:customer_id::INT, data:name::STRING, data:orders[0]:amount::DOUBLE
FROM events_variant;
```

**Advantages over from_json():**
- No schema required upfront
- Handles schema evolution automatically
- Faster for ad-hoc exploration

**Limitations:** Requires Databricks Runtime 15.0+ (not Community Edition)

⚠️ Conceptual — VARIANT not available on Community Edition.

---
# 📗 Section 8: JSON Lines (JSONL) Processing

In [0]:
import json

# JSONL: one JSON object per line (streamable, appendable)
jsonl_data = """{"event_id": 1, "type": "click", "user": "u001"}
{"event_id": 2, "type": "purchase", "user": "u002", "amount": 99.99}
{"event_id": 3, "type": "click", "user": "u001"}
{"INVALID JSON LINE
{"event_id": 4, "type": "signup", "user": "u003"}"""

# Process line by line (memory efficient for large files)
valid_events = []
errors = []

for line_num, line in enumerate(jsonl_data.strip().split("\n"), 1):
    try:
        event = json.loads(line)
        valid_events.append(event)
    except json.JSONDecodeError as e:
        errors.append({"line": line_num, "error": str(e), "raw": line[:50]})

print(f"Valid events: {len(valid_events)}")
print(f"Parse errors: {len(errors)}")
for err in errors:
    print(f"  Line {err['line']}: {err['error']}")

---
# 📗 Section 9: Building JSON for APIs

In [0]:
import json
from datetime import datetime
from decimal import Decimal

# Build API request payloads from DataFrame rows
sample_orders = spark.table("techmart_dw.orders").limit(5).toPandas()

class PipelineEncoder(json.JSONEncoder):
    """Custom encoder for pipeline data types."""
    def default(self, obj):
        if isinstance(obj, datetime):
            return obj.isoformat()
        if isinstance(obj, Decimal):
            return float(obj)
        if hasattr(obj, "item"):  # numpy types
            return obj.item()
        return super().default(obj)

# Convert rows to API payloads
payloads = []
for _, row in sample_orders.iterrows():
    payload = {
        "order_id": int(row["order_id"]),
        "customer_id": int(row["customer_id"]),
        "total_amount": float(row["total_amount"]),
        "status": row["status"],
        "metadata": {"source": "databricks_pipeline", "batch": "batch_001"}
    }
    payloads.append(payload)

print(f"Built {len(payloads)} API payloads")
print(f"Sample:\n{json.dumps(payloads[0], cls=PipelineEncoder, indent=2)}")

---
# 🚀 Section 10: Mini Projects

## Project 1: JSON Event Processor

In [0]:
# Complete JSON event processor with validation and flattening
import json
from pyspark.sql.functions import *
from pyspark.sql.types import *
import pandas as pd

# Simulate raw JSON events (inconsistent schemas)
raw_events = [
    '{"event_id": 1, "type": "purchase", "user_id": "u001", "data": {"amount": 99.99, "currency": "USD"}}',
    '{"event_id": 2, "type": "click", "user_id": "u002", "data": {"page": "/products"}}',
    '{"event_id": 3, "type": "purchase", "user_id": "u003", "data": {"amount": 150.0, "currency": "EUR", "items": 3}}',
    'INVALID JSON',
    '{"event_id": 5, "type": "signup", "user_id": "u004"}',  # missing "data" field
    '{"event_id": 6, "type": "purchase", "user_id": "u005", "data": {"amount": "free"}}',  # bad amount type
]

# Process: validate, parse, flatten
valid_records = []
quarantine = []

for i, raw in enumerate(raw_events):
    try:
        event = json.loads(raw)
        # Validate required fields
        assert "event_id" in event, "Missing event_id"
        assert "type" in event, "Missing type"
        assert "user_id" in event, "Missing user_id"
        
        # Flatten
        flat = {
            "event_id": event["event_id"],
            "event_type": event["type"],
            "user_id": event["user_id"],
            "data_amount": event.get("data", {}).get("amount") if isinstance(event.get("data", {}).get("amount"), (int, float)) else None,
            "data_currency": event.get("data", {}).get("currency"),
        }
        valid_records.append(flat)
    except (json.JSONDecodeError, AssertionError) as e:
        quarantine.append({"index": i, "error": str(e), "raw": raw[:80]})

print(f"Results: {len(valid_records)} valid, {len(quarantine)} quarantined")
print(f"\nValid:")
for r in valid_records:
    print(f"  {r}")
print(f"\nQuarantine:")
for q in quarantine:
    print(f"  Line {q['index']}: {q['error']}")

# Write valid to Delta
if valid_records:
    spark.createDataFrame(pd.DataFrame(valid_records)).write.format("delta").mode("overwrite").saveAsTable("techmart_dw.silver_json_events")
    print(f"\n✅ Written {len(valid_records)} events to silver_json_events")

---
# 🏆 Section 11: Interview Questions

## Q1: How do you handle semi-structured JSON in a data lake?

1. **Bronze:** Store raw JSON as-is (STRING column or raw files)
2. **Silver:** Parse with explicit schema (from_json), flatten, validate
3. **Gold:** Fully relational columns, no JSON remaining
4. Handle schema evolution by making new fields Optional with defaults

## Q2: Schema evolution in JSON data?

- New fields: add to schema as Optional (won't break existing records)
- Removed fields: keep in schema as Optional (old records still valid)
- Type changes: version the schema, route records to correct parser
- Use VARIANT type for truly unpredictable schemas

## Q3: Flatten deeply nested JSON efficiently?

- **Python:** Recursive flatten function with dot-notation keys
- **PySpark:** from_json → access nested with dot notation → explode arrays
- **Key:** Define schema explicitly (don't rely on inference for production)

## Q4: JSON vs Parquet?

| JSON | Parquet |
|---|---|
| Human-readable | Binary (not readable) |
| Schema-on-read | Schema embedded |
| Row-oriented | Columnar |
| Slow to query | Fast to query |
| Good for: APIs, configs | Good for: analytics, data lake |

## Q5: Double-encoded JSON?

Sometimes a JSON field contains a JSON *string* (escaped quotes):
`{"data": "{\"name\": \"John\"}"}` — the inner value is a string, not an object.
Fix: `json.loads(json.loads(raw)["data"])` — parse twice.

## Q6: VARIANT vs from_json()?

- **VARIANT:** No schema needed, handles evolution, great for exploration
- **from_json():** Explicit schema, type-safe, better for production pipelines
- Use VARIANT for Bronze/exploration, from_json for Silver/production

## Q7: Parse a 10GB JSON file?

Never load into memory. Use:
1. **JSONL format:** Process line by line
2. **ijson library:** Streaming SAX-style parser
3. **PySpark:** `spark.read.json("path")` — distributed parsing
4. **Chunked reading:** Split file, process in parallel

## Q8: JSON arrays with different schemas per element?

Use a Union approach: try parsing with each known schema.
Or use VARIANT type. Or flatten to the superset of all fields (missing = NULL).

---
# 📗 Section 5: Schema Validation & JSON Schema

JSON Schema is the standard for validating JSON structure in DE pipelines.
Use it to validate API responses, event payloads, and config files.

In [0]:
import json
import jsonschema
from jsonschema import validate, ValidationError as JsonSchemaError
from typing import Any, Dict, List, Tuple

# Define a JSON Schema for order events
ORDER_SCHEMA = {
    "$schema": "http://json-schema.org/draft-07/schema#",
    "title": "OrderEvent",
    "type": "object",
    "required": ["order_id", "customer_id", "amount", "status", "items"],
    "properties": {
        "order_id": {
            "type": "integer",
            "minimum": 1,
        },
        "customer_id": {
            "type": "integer",
            "minimum": 1,
        },
        "amount": {
            "type": "number",
            "minimum": 0,
            "exclusiveMinimum": True,
        },
        "status": {
            "type": "string",
            "enum": ["pending", "processing", "shipped", "delivered", "cancelled"],
        },
        "items": {
            "type": "array",
            "minItems": 1,
            "items": {
                "type": "object",
                "required": ["product_id", "quantity", "price"],
                "properties": {
                    "product_id": {"type": "integer", "minimum": 1},
                    "quantity": {"type": "integer", "minimum": 1},
                    "price": {"type": "number", "minimum": 0},
                },
                "additionalProperties": False,
            },
        },
        "metadata": {
            "type": "object",
            "additionalProperties": True,  # Allow any extra fields
        },
    },
    "additionalProperties": False,
}

def validate_json_schema(data: dict, schema: dict) -> Tuple[bool, List[str]]:
    """Validate data against JSON Schema. Returns (is_valid, errors)."""
    try:
        validate(instance=data, schema=schema)
        return True, []
    except JsonSchemaError as e:
        # Collect all errors (not just first)
        validator = jsonschema.Draft7Validator(schema)
        errors = [
            f"{'.'.join(str(p) for p in err.absolute_path) or 'root'}: {err.message}"
            for err in validator.iter_errors(data)
        ]
        return False, errors

# Test cases
test_orders = [
    {
        "order_id": 1, "customer_id": 42, "amount": 150.0,
        "status": "pending",
        "items": [{"product_id": 1, "quantity": 2, "price": 75.0}],
    },
    {
        "order_id": -1, "customer_id": 42, "amount": 0,  # Invalid: id<1, amount=0
        "status": "unknown",  # Invalid status
        "items": [],  # Invalid: empty
    },
    {
        "order_id": 2, "customer_id": 5, "amount": 200.0,
        "status": "shipped",
        "items": [{"product_id": 1, "quantity": 1, "price": 200.0, "extra": "field"}],  # Extra field
    },
]

for i, order in enumerate(test_orders):
    is_valid, errors = validate_json_schema(order, ORDER_SCHEMA)
    status = "✅ VALID" if is_valid else "❌ INVALID"
    print(f"Order {i+1}: {status}")
    for err in errors:
        print(f"  - {err}")

In [0]:
# --- JSON Schema Registry (simplified) ---
class SchemaRegistry:
    """
    Lightweight schema registry for managing JSON schemas.
    In production, use Confluent Schema Registry or AWS Glue Schema Registry.
    """
    
    def __init__(self):
        self._schemas: Dict[str, Dict[str, Any]] = {}
    
    def register(self, name: str, version: int, schema: dict):
        key = f"{name}:v{version}"
        self._schemas[key] = schema
        print(f"  📋 Registered schema: {key}")
    
    def get(self, name: str, version: int = None) -> dict:
        if version:
            return self._schemas.get(f"{name}:v{version}")
        # Get latest version
        matching = [k for k in self._schemas if k.startswith(f"{name}:")]
        if not matching:
            raise KeyError(f"No schema found for: {name}")
        latest = sorted(matching)[-1]
        return self._schemas[latest]
    
    def validate(self, name: str, data: dict, version: int = None) -> Tuple[bool, List[str]]:
        schema = self.get(name, version)
        return validate_json_schema(data, schema)
    
    def list_schemas(self):
        return list(self._schemas.keys())

# Use the registry
registry = SchemaRegistry()
registry.register("order_event", 1, ORDER_SCHEMA)

# Validate using registry
valid_order = {
    "order_id": 100, "customer_id": 1, "amount": 99.99,
    "status": "pending",
    "items": [{"product_id": 5, "quantity": 1, "price": 99.99}],
}
is_valid, errors = registry.validate("order_event", valid_order)
print(f"\nRegistry validation: {'✅ VALID' if is_valid else '❌ INVALID'}")
print(f"Available schemas: {registry.list_schemas()}")

## 5.2 Complex JSON Transformations

Real-world JSON from APIs is messy — deeply nested, inconsistent, and
full of arrays. Here are the patterns to handle it.

In [0]:
# --- Deep flattening with path tracking ---
def flatten_json(obj: Any, prefix: str = "", sep: str = ".") -> Dict[str, Any]:
    """
    Recursively flatten nested JSON to dot-notation keys.
    Arrays are indexed: items.0.price, items.1.price, etc.
    """
    result = {}
    
    if isinstance(obj, dict):
        for key, value in obj.items():
            new_key = f"{prefix}{sep}{key}" if prefix else key
            result.update(flatten_json(value, new_key, sep))
    elif isinstance(obj, list):
        for i, item in enumerate(obj):
            new_key = f"{prefix}{sep}{i}" if prefix else str(i)
            result.update(flatten_json(item, new_key, sep))
    else:
        result[prefix] = obj
    
    return result

# Test with complex nested JSON
complex_event = {
    "event_id": "evt_001",
    "user": {
        "id": 42,
        "profile": {
            "name": "Alice",
            "address": {"city": "Seattle", "country": "US"},
        },
    },
    "items": [
        {"product_id": 1, "qty": 2, "price": 29.99},
        {"product_id": 2, "qty": 1, "price": 149.99},
    ],
    "metadata": {"source": "web", "version": "2.1"},
}

flat = flatten_json(complex_event)
print("Flattened JSON:")
for k, v in flat.items():
    print(f"  {k}: {v}")

In [0]:
# --- Safe nested access patterns ---
from functools import reduce

def safe_get(obj: Any, *keys, default=None) -> Any:
    """
    Safely access nested dict/list values without KeyError/IndexError.
    
    Usage:
        safe_get(data, "user", "profile", "name")
        safe_get(data, "items", 0, "price")
    """
    try:
        return reduce(
            lambda acc, key: acc[key] if isinstance(acc, (dict, list)) else default,
            keys, obj
        )
    except (KeyError, IndexError, TypeError):
        return default

# Test
data = {
    "user": {"profile": {"name": "Alice", "age": 30}},
    "items": [{"id": 1, "price": 99.99}],
}

print(safe_get(data, "user", "profile", "name"))      # Alice
print(safe_get(data, "user", "profile", "email"))     # None (missing)
print(safe_get(data, "items", 0, "price"))            # 99.99
print(safe_get(data, "items", 5, "price"))            # None (index OOB)
print(safe_get(data, "missing", "key", default="N/A"))  # N/A

# --- Normalize inconsistent API responses ---
def normalize_api_response(raw: dict) -> dict:
    """
    Handle APIs that return data in different shapes.
    Some return {"data": {...}}, others return the object directly.
    """
    # Unwrap common envelope patterns
    if "data" in raw and isinstance(raw["data"], dict):
        data = raw["data"]
    elif "result" in raw:
        data = raw["result"]
    elif "payload" in raw:
        data = raw["payload"]
    else:
        data = raw
    
    return {
        "id": safe_get(data, "id") or safe_get(data, "order_id") or safe_get(data, "_id"),
        "amount": safe_get(data, "amount") or safe_get(data, "total") or safe_get(data, "value", default=0),
        "status": (safe_get(data, "status") or safe_get(data, "state") or "unknown").lower(),
        "created_at": safe_get(data, "created_at") or safe_get(data, "timestamp") or safe_get(data, "date"),
    }

# Test with different response shapes
responses = [
    {"data": {"id": 1, "amount": 100.0, "status": "PENDING", "created_at": "2024-01-15"}},
    {"order_id": 2, "total": 200.0, "state": "Completed", "timestamp": "2024-01-16"},
    {"_id": "abc", "value": 50.0, "status": "shipped", "date": "2024-01-17"},
]

print("\nNormalized responses:")
for r in responses:
    print(f"  {normalize_api_response(r)}")

---
# 📗 Section 6: Practice Exercises

In [0]:
# ============================================================
# EXERCISE 1: JSON Event Processor
# ============================================================
# Process a stream of JSON events:
# 1. Validate against schema
# 2. Flatten nested structure
# 3. Extract key fields
# 4. Route to valid/invalid queues

import random

def generate_events(n=20):
    """Generate mix of valid and invalid events."""
    events = []
    for i in range(n):
        if random.random() < 0.8:  # 80% valid
            events.append({
                "order_id": i + 1,
                "customer_id": random.randint(1, 100),
                "amount": round(random.uniform(10, 1000), 2),
                "status": random.choice(["pending", "shipped", "delivered"]),
                "items": [{"product_id": j, "quantity": 1, "price": 50.0}
                          for j in range(1, random.randint(2, 4))],
            })
        else:  # 20% invalid
            events.append({
                "order_id": -1,  # Invalid
                "customer_id": i,
                "amount": 0,     # Invalid
                "status": "unknown",  # Invalid
                "items": [],     # Invalid
            })
    return events

def process_event_stream(events: list) -> dict:
    valid_queue = []
    invalid_queue = []
    
    for event in events:
        is_valid, errors = validate_json_schema(event, ORDER_SCHEMA)
        
        if is_valid:
            # Flatten and enrich
            flat = flatten_json(event)
            flat["_processed_at"] = "2024-01-15T10:00:00Z"
            flat["_item_count"] = len(event.get("items", []))
            valid_queue.append(flat)
        else:
            invalid_queue.append({
                "_raw": event,
                "_errors": errors,
                "_rejected_at": "2024-01-15T10:00:00Z",
            })
    
    print(f"Processed {len(events)} events:")
    print(f"  ✅ Valid:   {len(valid_queue)}")
    print(f"  ❌ Invalid: {len(invalid_queue)}")
    return {"valid": valid_queue, "invalid": invalid_queue}

random.seed(42)
result = process_event_stream(generate_events(20))
print(f"\nSample valid event keys: {list(result['valid'][0].keys())[:8]}")
print(f"Sample invalid error: {result['invalid'][0]['_errors'][0] if result['invalid'] else 'none'}")

---
# 📗 Section 5: JSON in PySpark -- Semi-Structured Data

## Handling Nested JSON in Spark

```python
from pyspark.sql.functions import from_json, to_json, get_json_object, json_tuple, explode
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType

# Parse JSON string column
schema = StructType([
    StructField("order_id", IntegerType()),
    StructField("items", ArrayType(StructType([
        StructField("product_id", IntegerType()),
        StructField("quantity", IntegerType()),
        StructField("price", StringType()),
    ]))),
    StructField("metadata", StructType([
        StructField("source", StringType()),
        StructField("version", StringType()),
    ]))
])

df = spark.table("bronze.raw_orders")
parsed = df.withColumn("data", from_json(col("json_column"), schema))

# Access nested fields
parsed.select(
    col("data.order_id"),
    col("data.metadata.source"),
    explode(col("data.items")).alias("item")
).select(
    col("order_id"),
    col("source"),
    col("item.product_id"),
    col("item.quantity"),
)
```

## JSON Schema Validation

```python
import jsonschema

order_schema = {
    "type": "object",
    "required": ["order_id", "customer_id", "amount"],
    "properties": {
        "order_id": {"type": "integer", "minimum": 1},
        "customer_id": {"type": "integer"},
        "amount": {"type": "number", "minimum": 0},
        "status": {"type": "string", "enum": ["pending", "completed", "cancelled"]},
    }
}

def validate_json(data, schema):
    try:
        jsonschema.validate(data, schema)
        return True, None
    except jsonschema.ValidationError as e:
        return False, str(e.message)
```

In [0]:
# JSON patterns for data engineering
import json
from datetime import datetime

print("JSON Patterns for Data Engineering")
print("=" * 60)

# Pattern 1: Flatten nested JSON
def flatten_json(nested, prefix="", sep="_"):
    """Flatten a nested JSON object."""
    flat = {}
    for key, value in nested.items():
        new_key = f"{prefix}{sep}{key}" if prefix else key
        if isinstance(value, dict):
            flat.update(flatten_json(value, new_key, sep))
        elif isinstance(value, list):
            for i, item in enumerate(value):
                if isinstance(item, dict):
                    flat.update(flatten_json(item, f"{new_key}_{i}", sep))
                else:
                    flat[f"{new_key}_{i}"] = item
        else:
            flat[new_key] = value
    return flat

nested_order = {
    "order_id": 1001,
    "customer": {"id": 42, "name": "Alice", "city": "NYC"},
    "items": [
        {"product_id": 1, "qty": 2, "price": 29.99},
        {"product_id": 2, "qty": 1, "price": 49.99},
    ],
    "metadata": {"source": "web", "version": "2.1"}
}

flat = flatten_json(nested_order)
print("\n1. Flattened JSON:")
for key, value in flat.items():
    print(f"   {key}: {value}")

# Pattern 2: Extract specific fields safely
def safe_get(data, *keys, default=None):
    """Safely navigate nested JSON."""
    current = data
    for key in keys:
        if isinstance(current, dict):
            current = current.get(key, default)
        elif isinstance(current, list) and isinstance(key, int):
            current = current[key] if key < len(current) else default
        else:
            return default
    return current

print("\n2. Safe nested access:")
print(f"   customer.name: {safe_get(nested_order, 'customer', 'name')}")
print(f"   items[0].price: {safe_get(nested_order, 'items', 0, 'price')}")
print(f"   missing.field: {safe_get(nested_order, 'missing', 'field', default='N/A')}")

# Pattern 3: JSON Lines (JSONL) -- common in DE
print("\n3. JSON Lines format (one JSON per line):")
records = [
    {"id": 1, "event": "click", "ts": "2024-03-15T10:00:00Z"},
    {"id": 2, "event": "purchase", "ts": "2024-03-15T10:01:00Z"},
    {"id": 3, "event": "view", "ts": "2024-03-15T10:02:00Z"},
]
jsonl = "\n".join(json.dumps(r) for r in records)
print(jsonl)
print(f"   Parsed back: {len([json.loads(line) for line in jsonl.splitlines()])} records")


---
# ✅ Notebook Complete!

**What was covered:**
- Python json module: loads/dumps, error handling, custom encoders
- Nested JSON navigation: safe .get() access, recursive flattening
- PySpark JSON: from_json, get_json_object, explode arrays
- Flattening nested structures into relational columns
- VARIANT type concepts (Databricks 2024+)
- JSONL streaming processing
- Building API payloads from DataFrames
- Mini project: complete JSON event processor with validation
- 8 interview questions

**Next:** Notebook 16 — Regex for Data Engineering

---